In [1]:
print("Here begins the Bayesian networks notebook.")

Here begins the Bayesian networks notebook.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/student_depression_bn_clean.csv")

df_bn.head()

,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,male,33.0,visakhapatnam,student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,bachelor,yes,3.0,1.0,no,yes
1,female,24.0,bangalore,student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,bachelor,no,3.0,2.0,yes,no
2,male,31.0,srinagar,student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,bachelor,no,9.0,1.0,yes,no
3,female,28.0,varanasi,student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,bachelor,yes,4.0,5.0,yes,yes
4,female,25.0,jaipur,student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,master,yes,1.0,1.0,no,no


In [2]:
# Inspect dataset shape, column types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(27901, 17)
Gender                                       str
Age                                      float64
City                                         str
Profession                                   str
Academic Pressure                        float64
Work Pressure                            float64
CGPA                                     float64
Study Satisfaction                       float64
Job Satisfaction                         float64
Sleep Duration                               str
Dietary Habits                               str
Degree                                       str
Have you ever had suicidal thoughts ?        str
Work/Study Hours                         float64
Financial Stress                         float64
Family History of Mental Illness             str
Depression                                   str
dtype: object

Missing values per column:
CGPA                                     9
Financial Stress                         3
Gender                     

In [3]:
# Convert all columns to string
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Gender                                   string
Age                                      string
City                                     string
Profession                               string
Academic Pressure                        string
Work Pressure                            string
CGPA                                     string
Study Satisfaction                       string
Job Satisfaction                         string
Sleep Duration                           string
Dietary Habits                           string
Degree                                   string
Have you ever had suicidal thoughts ?    string
Work/Study Hours                         string
Financial Stress                         string
Family History of Mental Illness         string
Depression                               string
dtype: object


In [5]:
# Check the number of unique states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
6,CGPA,331
2,City,52
1,Age,34
3,Profession,14
13,Work/Study Hours,13
11,Degree,6
4,Academic Pressure,6
7,Study Satisfaction,6
14,Financial Stress,5
9,Sleep Duration,5


In [6]:
# Discretize CGPA into 5 quantile bins
df_bn["CGPA_binned"] = pd.qcut(
    pd.to_numeric(df_bn["CGPA"], errors="coerce"),
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
).astype("string")

# Discretize Age into 5 quantile bins
df_bn["Age_binned"] = pd.qcut(
    pd.to_numeric(df_bn["Age"], errors="coerce"),
    q=5,
    labels=["very_young", "young", "mid", "older", "old"],
    duplicates="drop"
).astype("string")

# Drop original continuous columns and City
df_bn = df_bn.drop(columns=["CGPA", "City", "Age"])

# Filter to students only — consistent with ML notebooks
# Non-student professions (0.1% of data) represent a different
# population context and are dropped for consistency
non_student = (df_bn["Profession"] != "student").sum()
total = len(df_bn)
print(f"Non-student rows dropped: {non_student} ({non_student/total:.1%})")

df_bn = df_bn[df_bn["Profession"] == "student"].copy()
df_bn = df_bn.drop(columns=["Profession"])

# Drop Work Pressure — values (0, 2, 5) indicate data entry
# inconsistency and the variable is not applicable to students
# Drop Job Satisfaction — not applicable to a student population
df_bn = df_bn.drop(columns=["Work Pressure", "Job Satisfaction"], errors="ignore")

df_bn_model = df_bn.copy()

print(f"Shape after filtering: {df_bn_model.shape}")
print(df_bn_model.columns.tolist())

# Verify cardinality after cleaning
cardinality = pd.DataFrame({
    "Variable": df_bn_model.columns,
    "Num_States": [df_bn_model[col].nunique(dropna=True) for col in df_bn_model.columns]
}).sort_values("Num_States", ascending=False)

print(cardinality)

Non-student rows dropped: 31 (0.1%)
Shape after filtering: (27870, 13)
['Gender', 'Academic Pressure', 'Study Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression', 'CGPA_binned', 'Age_binned']
                                 Variable  Num_States
7                        Work/Study Hours          13
2                      Study Satisfaction           6
1                       Academic Pressure           6
5                                  Degree           6
3                          Sleep Duration           5
11                            CGPA_binned           5
12                             Age_binned           5
8                        Financial Stress           5
4                          Dietary Habits           4
0                                  Gender           2
6   Have you ever had suicidal thoughts ?           2
10                   

In [7]:
# Bin Work/Study Hours into 4 groups to reduce sparsity
df_bn_model["Work/Study Hours"] = pd.cut(
    pd.to_numeric(df_bn_model["Work/Study Hours"], errors="coerce"),
    bins=[-1, 2, 5, 8, 12],
    labels=["low", "moderate", "high", "very_high"]
).astype("string")

print(df_bn_model["Work/Study Hours"].value_counts())

# Verify cardinality after cleaning
cardinality = pd.DataFrame({
    "Variable": df_bn_model.columns,
    "Num_States": [df_bn_model[col].nunique(dropna=True) for col in df_bn_model.columns]
}).sort_values("Num_States", ascending=False)

print(cardinality)

Work/Study Hours
very_high    12312
high          6756
low           4433
moderate      4369
Name: count, dtype: Int64
                                 Variable  Num_States
1                       Academic Pressure           6
2                      Study Satisfaction           6
5                                  Degree           6
8                        Financial Stress           5
3                          Sleep Duration           5
11                            CGPA_binned           5
12                             Age_binned           5
7                        Work/Study Hours           4
4                          Dietary Habits           4
0                                  Gender           2
6   Have you ever had suicidal thoughts ?           2
10                             Depression           2
9        Family History of Mental Illness           2


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import HillClimbSearch, ExpertKnowledge, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [9]:
# Unconstrained Bayesian Network structure using Hill-Climb Search
# - bic-d - used for categorical data
# - max_indegree - the number of parents per node is limited to avoid dense graphs
hc = HillClimbSearch(df_bn_model)

learned_dag = hc.estimate(
    scoring_method="bic-d",
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag.nodes()))
print("Number of edges:", len(learned_dag.edges()))
print("\nLearned edges:")
print(sorted(learned_dag.edges()))

Number of nodes: 13
Number of edges: 14

Learned edges:
[('Age_binned', 'Depression'), ('CGPA_binned', 'Gender'), ('Degree', 'Age_binned'), ('Depression', 'Academic Pressure'), ('Depression', 'Dietary Habits'), ('Depression', 'Family History of Mental Illness'), ('Depression', 'Financial Stress'), ('Depression', 'Have you ever had suicidal thoughts ?'), ('Depression', 'Sleep Duration'), ('Depression', 'Study Satisfaction'), ('Depression', 'Work/Study Hours'), ('Gender', 'Degree'), ('Gender', 'Dietary Habits'), ('Study Satisfaction', 'Academic Pressure')]


In [24]:
NON_CAUSED = [
    "Gender", "Age_binned", "Family History of Mental Illness",
    "Academic Pressure", "Financial Stress", "Sleep Duration",
    "Dietary Habits", "Work/Study Hours", "CGPA_binned", "Degree"
]

implausible = [e for e in learned_dag.edges() if e[1] in NON_CAUSED]

print(f"Implausible edges in unconstrained BN: {len(implausible)}")
print("\nEdge | Issue")
print("-" * 60)
for src, tgt in sorted(implausible):
    if tgt in ["Gender", "Age_binned", "Family History of Mental Illness"]:
        reason = "demographic/background variable cannot be caused"
    else:
        reason = "predictor variable should not be caused by outcome"
    print(f"  {src} → {tgt} | {reason}")

Implausible edges in unconstrained BN: 11

Edge | Issue
------------------------------------------------------------
  CGPA_binned → Gender | demographic/background variable cannot be caused
  Degree → Age_binned | demographic/background variable cannot be caused
  Depression → Academic Pressure | predictor variable should not be caused by outcome
  Depression → Dietary Habits | predictor variable should not be caused by outcome
  Depression → Family History of Mental Illness | demographic/background variable cannot be caused
  Depression → Financial Stress | predictor variable should not be caused by outcome
  Depression → Sleep Duration | predictor variable should not be caused by outcome
  Depression → Work/Study Hours | predictor variable should not be caused by outcome
  Gender → Degree | predictor variable should not be caused by outcome
  Gender → Dietary Habits | predictor variable should not be caused by outcome
  Study Satisfaction → Academic Pressure | predictor variable sho

In [10]:
# Convert the learned graph into a discrete Bayesian Network
bn_unconstrained = DiscreteBayesianNetwork(learned_dag.edges())

# Fit CPDs through Bayesian parameter estimation
# BDeu prior avoids zero probabilities
bn_unconstrained.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_unconstrained.check_model())

Model check: True


In [11]:
# conditional probability distribution of the target variable
# how Depression depends on learned parent variables

cpd_dep = bn_unconstrained.get_cpds("Depression")

# Extract state names
dep_states = cpd_dep.state_names["Depression"]
parent = list(cpd_dep.state_names.keys())[1]
parent_states = cpd_dep.state_names[parent]

# Values
values = cpd_dep.values

# DataFrame
cpd_df = pd.DataFrame(
    values,
    index=[f"Depression = {s}" for s in dep_states],
    columns=[f"{parent} = {s}" for s in parent_states]
)

cpd_df

,Age_binned = mid,Age_binned = old,Age_binned = older,Age_binned = very_young,Age_binned = young
Depression = no,0.39712,0.613753,0.500111,0.287945,0.356535
Depression = yes,0.60288,0.386247,0.499889,0.712055,0.643465


In [12]:
# Create inference engine for the unconstrained BN
infer_unconstrained = VariableElimination(bn_unconstrained)  # type: ignore

In [13]:
# Query 1:
# Probability of Depression given high academic pressure and suicidal thoughts
q1 = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Academic Pressure": "5.0",
        "Have you ever had suicidal thoughts ?": "yes"
    },
    show_progress=False
)

print(q1)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0572 |
+-----------------+-------------------+
| Depression(yes) |            0.9428 |
+-----------------+-------------------+


In [14]:
# Query 2:
# Probability of Depression under poor sleep, unhealthy diet, and financial stress
q2 = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Sleep Duration": "Less than 5 hours",
        "Dietary Habits": "Unhealthy",
        "Financial Stress": "5.0"
    },
    show_progress=False
)

print(q2)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0950 |
+-----------------+-------------------+
| Depression(yes) |            0.9050 |
+-----------------+-------------------+


In [15]:
# Query 3:
# Depression probabilities under two contrasting scenarios

q3_supportive = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Study Satisfaction": "5.0",
        "Sleep Duration": "7-8 hours",
        "Financial Stress": "1.0"
    },
    show_progress=False
)

q3_risky = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Study Satisfaction": "1.0",
        "Sleep Duration": "Less than 5 hours",
        "Financial Stress": "5.0"
    },
    show_progress=False
)

print("Depression distribution under supportive conditions:")
print(q3_supportive)

print("\nDepression distribution under unsupportive conditions:")
print(q3_risky)

Depression distribution under supportive conditions:
+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.7649 |
+-----------------+-------------------+
| Depression(yes) |            0.2351 |
+-----------------+-------------------+

Depression distribution under unsupportive conditions:
+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0948 |
+-----------------+-------------------+
| Depression(yes) |            0.9052 |
+-----------------+-------------------+


In [16]:
# Build a restricted BN with simple expert constraints- prevent obviously implausible directions

forbidden_edges = [
    # Depression should not cause background variables
    ("Depression", "Gender"),
    ("Depression", "Age"),
    ("Depression", "City"),
    ("Depression", "Profession"),
    ("Depression", "Degree"),
    ("Depression", "Family History of Mental Illness"),

    # Depression should not cause broad structural or background study/work context
    ("Depression", "Academic Pressure"),
    ("Depression", "Work Pressure"),
    ("Depression", "CGPA"),
    ("Depression", "Study Satisfaction"),
    ("Depression", "Job Satisfaction"),
    ("Depression", "Sleep Duration"),
    ("Depression", "Dietary Habits"),
    ("Depression", "Work/Study Hours"),
    ("Depression", "Financial Stress"),

    # Suicidal thoughts should not cause background variables
    ("Have you ever had suicidal thoughts?", "Gender"),
    ("Have you ever had suicidal thoughts?", "Age"),
    ("Have you ever had suicidal thoughts?", "Family History of Mental Illness"),
]

expert_knowledge = ExpertKnowledge(
    forbidden_edges=forbidden_edges
)

hc_restricted = HillClimbSearch(df_bn_model)

learned_dag_restricted = hc_restricted.estimate(
    scoring_method="bic-d",
    expert_knowledge=expert_knowledge,
    max_indegree=3,
    show_progress=False
)

print("Restricted BN edges:")
print(sorted(learned_dag_restricted.edges()))

Restricted BN edges:
[('Academic Pressure', 'Depression'), ('Academic Pressure', 'Financial Stress'), ('Age_binned', 'Degree'), ('CGPA_binned', 'Gender'), ('CGPA_binned', 'Study Satisfaction'), ('Depression', 'Age_binned'), ('Depression', 'Have you ever had suicidal thoughts ?'), ('Financial Stress', 'Depression'), ('Gender', 'Dietary Habits'), ('Have you ever had suicidal thoughts ?', 'Dietary Habits'), ('Have you ever had suicidal thoughts ?', 'Family History of Mental Illness'), ('Have you ever had suicidal thoughts ?', 'Sleep Duration'), ('Have you ever had suicidal thoughts ?', 'Work/Study Hours'), ('Study Satisfaction', 'Academic Pressure')]


In [25]:
unconstrained_edges = set(learned_dag.edges())
restricted_edges    = set(learned_dag_restricted.edges())

shared  = unconstrained_edges & restricted_edges
removed = unconstrained_edges - restricted_edges
added   = restricted_edges - unconstrained_edges

print(f"Unconstrained BN — total edges : {len(unconstrained_edges)}")
print(f"Restricted BN    — total edges : {len(restricted_edges)}")
print(f"Shared edges                   : {len(shared)}")
print(f"Removed by constraints         : {len(removed)}")
print(f"New edges in restricted BN     : {len(added)}")

print("\n--- Edges removed by expert constraints ---")
for src, tgt in sorted(removed):
    print(f"  {src} → {tgt}")

print("\n--- New edges introduced in restricted BN ---")
for src, tgt in sorted(added):
    print(f"  {src} → {tgt}")

print("\n--- Shared edges (stable across both BNs) ---")
for src, tgt in sorted(shared):
    print(f"  {src} → {tgt}")

Unconstrained BN — total edges : 14
Restricted BN    — total edges : 14
Shared edges                   : 4
Removed by constraints         : 10
New edges in restricted BN     : 10

--- Edges removed by expert constraints ---
  Age_binned → Depression
  Degree → Age_binned
  Depression → Academic Pressure
  Depression → Dietary Habits
  Depression → Family History of Mental Illness
  Depression → Financial Stress
  Depression → Sleep Duration
  Depression → Study Satisfaction
  Depression → Work/Study Hours
  Gender → Degree

--- New edges introduced in restricted BN ---
  Academic Pressure → Depression
  Academic Pressure → Financial Stress
  Age_binned → Degree
  CGPA_binned → Study Satisfaction
  Depression → Age_binned
  Financial Stress → Depression
  Have you ever had suicidal thoughts ? → Dietary Habits
  Have you ever had suicidal thoughts ? → Family History of Mental Illness
  Have you ever had suicidal thoughts ? → Sleep Duration
  Have you ever had suicidal thoughts ? → Work/S

In [17]:
# Fit the restricted BN
bn_restricted = DiscreteBayesianNetwork(learned_dag_restricted.edges())

bn_restricted.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Restricted model check:", bn_restricted.check_model())

Restricted model check: True


In [18]:
# Compare the direct parent variables of Depression in the restricted BN
if "Depression" in bn_restricted.nodes():
    print("Parents of Depression in restricted BN:")
    print(list(bn_restricted.predecessors("Depression")))

Parents of Depression in restricted BN:
['Academic Pressure', 'Financial Stress']


In [19]:
# Create inference engine for the restricted BN
infer_restricted = VariableElimination(bn_restricted)  # type: ignore

In [20]:
# Query 4:
# Probability of Depression given suicidal thoughts and family history
q4 = infer_restricted.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Have you ever had suicidal thoughts ?": "yes",
        "Family History of Mental Illness": "yes"
    },
    show_progress=False
)

print(q4)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.2098 |
+-----------------+-------------------+
| Depression(yes) |            0.7902 |
+-----------------+-------------------+


In [21]:
# Query 5:
# Probability of Depression under strong academic and financial pressure
q5 = infer_restricted.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Academic Pressure": "5.0",
        "Financial Stress": "5.0",
        "Study Satisfaction": "1.0"
    },
    show_progress=False
)

print(q5)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0460 |
+-----------------+-------------------+
| Depression(yes) |            0.9540 |
+-----------------+-------------------+


In [22]:
# Summarize the local neighborhood of the target variable
summary_rows = []

if "Depression" in bn_restricted.nodes():
    summary_rows.append({
        "Target": "Depression",
        "Parents": list(bn_restricted.predecessors("Depression")),
        "Children": list(bn_restricted.successors("Depression"))
    })

pd.DataFrame(summary_rows)

,Target,Parents,Children
0,Depression,"[Academic Pressure, Financial Stress]","[Have you ever had suicidal thoughts ?, Age_bi..."


In [23]:
# Inspect the CPD of Depression in the restricted model
if "Depression" in bn_restricted.nodes():
    print("Restricted CPD for Depression:")
    print(bn_restricted.get_cpds("Depression"))

Restricted CPD for Depression:
+-------------------+-----+------------------------+
| Academic Pressure | ... | Academic Pressure(5.0) |
+-------------------+-----+------------------------+
| Financial Stress  | ... | Financial Stress(5.0)  |
+-------------------+-----+------------------------+
| Depression(no)    | ... | 0.04600800139154636    |
+-------------------+-----+------------------------+
| Depression(yes)   | ... | 0.9539919986084536     |
+-------------------+-----+------------------------+


In [26]:
import numpy as np

def bic_local_score(node, parents, data):
    n = len(data.dropna(subset=[node] + parents))
    node_states = data[node].dropna().unique()
    r_i = len(node_states)

    if len(parents) == 0:
        q_i = 1
        counts = data[node].value_counts()
        ll = sum(c * np.log(c / counts.sum()) for c in counts if c > 0)
        k = q_i * (r_i - 1)
    else:
        parent_states = data[parents + [node]].dropna()
        grouped = parent_states.groupby(parents)
        q_i = len(grouped)
        ll = 0
        for _, group in grouped:
            total = len(group)
            if total == 0:
                continue
            counts = group[node].value_counts()
            ll += sum(c * np.log(c / total) for c in counts if c > 0)
        k = q_i * (r_i - 1)

    return ll - (k / 2) * np.log(n)

def compute_total_bic(dag, data):
    return sum(
        bic_local_score(node, list(dag.predecessors(node)), data)
        for node in dag.nodes()
    )

score_unconstrained = compute_total_bic(bn_unconstrained, df_bn_model)
score_restricted    = compute_total_bic(bn_restricted, df_bn_model)

diff = score_restricted - score_unconstrained
pct  = abs(diff / score_unconstrained) * 100

print(f"BIC score — Unconstrained BN : {score_unconstrained:.2f}")
print(f"BIC score — Restricted BN    : {score_restricted:.2f}")
print(f"Difference (restricted - unconstrained) : {diff:.2f}")
print(f"Relative difference          : {pct:.2f}%")
print()
if diff < 0:
    print("The restricted BN has a lower (worse) BIC score.")
    print(f"Expert constraints cost {pct:.2f}% in statistical fit.")
else:
    print("The restricted BN has an equal or better BIC score.")
    print("Expert constraints did not reduce statistical fit.")

BIC score — Unconstrained BN : -420984.73
BIC score — Restricted BN    : -422192.26
Difference (restricted - unconstrained) : -1207.53
Relative difference          : 0.29%

The restricted BN has a lower (worse) BIC score.
Expert constraints cost 0.29% in statistical fit.


In [27]:
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np

infer_restricted = VariableElimination(bn_restricted)

TARGET = "Depression"
evidence_cols = [c for c in df_bn_model.columns if c != TARGET]
df_pred = df_bn_model.dropna(subset=[TARGET]).copy()

y_true    = []
y_pred_bn = []
y_prob_bn = []
state_order = ["no", "yes"]

for _, row in df_pred.iterrows():
    evidence = {
        col: str(row[col])
        for col in evidence_cols
        if pd.notna(row[col]) and str(row[col]) not in ["<NA>", "nan", "None"]
    }
    try:
        result = infer_restricted.query(
            variables=[TARGET],
            evidence=evidence,
            show_progress=False
        )
        states = result.state_names[TARGET]
        probs  = result.values
        pred   = states[np.argmax(probs)]

        prob_row = [
            probs[states.index(s)] if s in states else 0.0
            for s in state_order
        ]

        y_true.append(str(row[TARGET]))
        y_pred_bn.append(pred)
        y_prob_bn.append(prob_row)
    except Exception:
        continue

y_prob_bn = np.array(y_prob_bn)

print(f"Predictions made: {len(y_true)} / {len(df_pred)}")
print()
print(classification_report(y_true, y_pred_bn, labels=state_order))

macro_f1_bn    = f1_score(y_true, y_pred_bn, average="macro", labels=state_order)
weighted_f1_bn = f1_score(y_true, y_pred_bn, average="weighted", labels=state_order)

print(f"Macro F1    : {macro_f1_bn:.3f}")
print(f"Weighted F1 : {weighted_f1_bn:.3f}")

# AUC — binary, use probability of positive class
try:
    y_true_bin = [1 if v == "yes" else 0 for v in y_true]
    auc_bn = roc_auc_score(y_true_bin, y_prob_bn[:, 1])
    print(f"AUC         : {auc_bn:.3f}")
except Exception as e:
    print(f"AUC could not be computed: {e}")

Predictions made: 27870 / 27870

              precision    recall  f1-score   support

          no       0.81      0.77      0.79     11562
         yes       0.85      0.87      0.86     16308

    accuracy                           0.83     27870
   macro avg       0.83      0.82      0.83     27870
weighted avg       0.83      0.83      0.83     27870

Macro F1    : 0.826
Weighted F1 : 0.832
AUC         : 0.906


In [28]:
comparison = pd.DataFrame([
    {"Model": "Logistic Regression",       "Type": "Baseline ML", "CV Macro F1": 0.841, "CV F1 std": 0.005, "AUC": 0.921},
    {"Model": "Decision Tree",             "Type": "Baseline ML", "CV Macro F1": 0.822, "CV F1 std": 0.006, "AUC": 0.901},
    {"Model": "Random Forest",             "Type": "Baseline ML", "CV Macro F1": 0.837, "CV F1 std": 0.005, "AUC": 0.917},
    {"Model": "XGBoost",                   "Type": "Baseline ML", "CV Macro F1": 0.841, "CV F1 std": 0.004, "AUC": 0.921},
    {"Model": "RF Selection → LogReg",     "Type": "Hybrid ML",   "CV Macro F1": 0.839, "CV F1 std": 0.004, "AUC": 0.921},
    {"Model": "XGB Selection → LogReg",    "Type": "Hybrid ML",   "CV Macro F1": 0.840, "CV F1 std": 0.005, "AUC": 0.921},
    {"Model": "Stacking (LR + RF + XGB)",  "Type": "Hybrid ML",   "CV Macro F1": 0.840, "CV F1 std": 0.005, "AUC": 0.921},
    {
        "Model": "Bayesian Network (restricted)",
        "Type": "BN",
        "CV Macro F1": round(macro_f1_bn, 3),
        "CV F1 std": None,
        "AUC": round(auc_bn, 3) if "auc_bn" in dir() else None,
    },
]).set_index("Model")

print(comparison.to_string())

                                      Type  CV Macro F1  CV F1 std    AUC
Model                                                                    
Logistic Regression            Baseline ML        0.841      0.005  0.921
Decision Tree                  Baseline ML        0.822      0.006  0.901
Random Forest                  Baseline ML        0.837      0.005  0.917
XGBoost                        Baseline ML        0.841      0.004  0.921
RF Selection → LogReg            Hybrid ML        0.839      0.004  0.921
XGB Selection → LogReg           Hybrid ML        0.840      0.005  0.921
Stacking (LR + RF + XGB)         Hybrid ML        0.840      0.005  0.921
Bayesian Network (restricted)           BN        0.826        NaN  0.906


In [ ]:
## Bayesian Network — Results Summary (Student Depression Dataset)

### Structure Learning
# Two BNs were learned using Hill-Climb Search with BIC-d scoring:
# - Unconstrained BN: 14 edges, no domain restrictions
# - Restricted BN: 14 edges, forbidden edges encoding causal domain
#   knowledge about depression predictors

### Implausible Edges (Unconstrained BN)
# 11 of 14 edges (79%) were flagged as causally implausible in the
# unconstrained BN, including Depression → Academic Pressure,
# Depression → Financial Stress, and Depression → Sleep Duration.
# This is a striking result — the unconstrained learner produced
# an almost entirely reverse-causal structure, treating depression
# as a cause of its own predictors. This directly confirms RQ4:
# data-driven structure learning alone cannot recover causal
# direction without domain constraints.

### Structural Stability
# Only 4 of 14 unconstrained edges (29%) were preserved in the
# restricted BN — the lowest stability rate across all datasets.
# This indicates the unconstrained structure was fundamentally
# misspecified. The shared edges are:
# - CGPA_binned → Gender (spurious association)
# - Depression → Have you ever had suicidal thoughts ? (retained
#   in restricted BN — direction is debatable, see note below)
# - Gender → Dietary Habits
# - Study Satisfaction → Academic Pressure

### BIC Score
# The restricted BN incurs only a 0.29% BIC penalty — the smallest
# cost across all datasets. This means the expert-constrained
# structure is nearly as statistically optimal as the unconstrained
# one, while being far more causally plausible. This is the
# strongest argument across all datasets for using domain
# constraints in BN structure learning.

### Causal Structure (Restricted BN)
# - Parents of Depression: Academic Pressure, Financial Stress
# - Children of Depression: Have you ever had suicidal thoughts?,
#   Age_binned
# This structure is partially plausible — academic pressure and
# financial stress as causes of depression is well-supported by
# the literature. However Depression → suicidal thoughts retained
# as a child edge is worth noting: suicidal ideation is more
# accurately a symptom co-occurring with depression rather than
# a downstream consequence. This edge direction is consistent
# with the co-symptom interpretation flagged in the ML notebooks.

### Important Note on CGPA_binned → Gender
# This edge is spurious and reflects a statistical association
# in the data rather than a causal relationship. It was not
# covered by the forbidden edges list and persists in both BNs.
# It should be added to the forbidden edges in a future revision
# or acknowledged as a known artifact.

### Predictive Performance
# The BN achieves Macro F1: 0.826 and AUC: 0.906 on the full
# observed dataset. This is lower than all ML baselines
# (best ML AUC: 0.921), unlike MH Tech where BN AUC exceeded
# ML. The gap is consistent with the heavily misspecified
# unconstrained structure — the restricted BN recovers a
# reasonable but simplified causal skeleton that sacrifices
# some discriminative power for interpretability.

### Key Findings for RQ1-RQ4
# - RQ1: BN Macro F1 (0.826) is below ML baselines (~0.837–0.841),
#   confirming that for well-structured individual-level data,
#   discriminative ML models outperform generative BN models
#   on prediction
# - RQ2: Restricted BN CPD table explicitly shows P(Depression |
#   Academic Pressure, Financial Stress) — more causally
#   interpretable than feature importance scores
# - RQ3: Restricted BN recovers Academic Pressure + Financial
#   Stress → Depression as the core causal pathway, consistent
#   with psychological literature on student mental health
# - RQ4: 79% of unconstrained edges were implausible — the
#   strongest evidence across all datasets that purely
#   data-driven BN structure learning fails without domain
#   knowledge